In [1]:
import copy
import csv
import re

In [2]:
fname = "Temporal_final.csv"

In [3]:
with open(fname, mode='r', newline='', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    records = [row for row in reader]

In [4]:
len(records)

2495

In [5]:
records[0]

{'': '0',
 'id': '4222362',
 'subj': 'George Rankin',
 'prop': 'occupation',
 'obj': 'politician',
 'subj_id': 'Q5543720',
 'prop_id': 'P106',
 'obj_id': 'Q82955',
 's_pop': '142',
 'o_pop': '25692',
 'question': "What is George Rankin's occupation?",
 'possible_answers': '["politician", "political leader", "political figure", "polit.", "pol"]',
 'num_edits': '4.0',
 'context': ' George James Rankin,  (1 May 1887 – 28 December 1957) was a [ENTITY]. He served in both the House of Representatives and the Senate, representing the Country Party of Australia.',
 'replace_quality': '0.718457043170929',
 'replace_name': 'judge',
 's_pop_new': '179.0',
 'o_pop_new': '3.0'}

In [6]:
templates = ["What is [X]'s occupation?",
            "In what city was [X] born?",
            "What genre is [X]?",
            "Who is the father of [X]?",
            "In what country is [X]?",
            "Who was the producer of [X]?",
            "Who was the director of [X]?",
            "What is [X] the capital of?",
            "Who was the screenwriter for [X]?",
            "Who was the composer of [X]?",
            "What color is [X]?",
            "What is the religion of [X]?",
            "What sport does [X] play?",
            "Who is the author of [X]?",
            "Who is the mother of [X]?",
            "What is the capital of [X]?"]

In [7]:
def template_to_regex(template, placeholder='[X]'):
    # Escape special characters (e.g. '?', "'", etc.)
    pattern = re.escape(template)
    # Replace all occurrences of the placeholder
    pattern = pattern.replace(re.escape(placeholder), r"(.+)")
    return "^" + pattern + "$"

def match_template(sentence, templates, placeholder='[X]'):
    for template in templates:
        regex = template_to_regex(template, placeholder)
        match = re.match(regex, sentence)
        if match:
            return template, match.groups()
    return None, None

In [8]:
sentence = "What is Herlyn Espinal's occupation?"

matched_template, extracted = match_template(sentence, templates)

print("Matched Template:", matched_template)
print("Extracted Value(s):", extracted)

Matched Template: What is [X]'s occupation?
Extracted Value(s): ('Herlyn Espinal',)


In [9]:
extracted_records = []
remaining_records = copy.deepcopy(records)

for r in records[:]:
    matched_template, extracted = match_template(r["question"], templates)
    if matched_template and extracted:
        remaining_records.remove(r)
        r["matched_template"] = matched_template
        r["extracted_sub"] = extracted
        extracted_records.append(r) 

In [10]:
print(f"{len(records)=}")
print(f"{len(extracted_records)=}")
print(f"{len(remaining_records)=}")

len(records)=2495
len(extracted_records)=2495
len(remaining_records)=0


In [11]:
for r in remaining_records[:10]:
    print(r["question"])

In [12]:
filename = "Temporal_extracted_templates.csv"

# Writing to CSV
with open(filename, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=extracted_records[0].keys())
    writer.writeheader()  # Write the header row
    writer.writerows(extracted_records)  # Write the data